# RoadSage — Phase 2: Lane Detection Pipeline Validation

## Objective
Validate the end-to-end lane detection pipeline on real MNNIT campus road images.  
This notebook is the **Phase 2 exit gate** — it must produce clean visual outputs
and pass all automated checks in the final cell before Phase 3 begins.

## What This Notebook Tests
| # | Test | Expected |
|---|------|----------|
| 1 | UFLDv2 ONNX inference | Runs on every image without crash |
| 2 | BEV perspective warp | 3×3 matrix, top-down road appearance |
| 3 | Lane geometry (polynomial fit) | Offset, curvature, width in metric units |
| 4 | Full visualisation overlay | Annotated frames saved to `outputs/` |
| 5 | Dataset coverage | ≥ 30 % of frames have both lane markings |
| 6 | CPU inference latency | P95 < 100 ms, target < 30 ms |

## How to Run
1. **Activate venv** — `venv\Scripts\activate` (Windows) or `source venv/bin/activate` (Linux/macOS)
2. **Check model weights** — `bash models/download_models.sh` (if not already downloaded)
3. **Run all cells** — *Kernel → Restart & Run All*
4. **Inspect the final cell** for the Pass / Fail verdict

In [ ]:
import os
import sys
from pathlib import Path

# Ensure working directory is project root so all relative paths resolve correctly
cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import random

from app.lane_detection.ufld_model import UFLDLaneDetector, LaneDetectionResult
from app.lane_detection.bev_transform import BEVTransform, BEVConfig
from app.lane_detection.lane_geometry import LaneGeometryComputer, make_empty_geometry
from app.explainability.visualizer import create_full_visualization, VisualizationConfig
import yaml

plt.ioff()  # prevent blocking pop-ups in notebook output

config = yaml.safe_load(open("configs/production.yaml"))
os.makedirs("outputs", exist_ok=True)
print("All imports successful")

In [ ]:
detector = UFLDLaneDetector(config_path="configs/lane_detection.yaml")
bev_config = BEVConfig.from_yaml("configs/lane_detection.yaml")
bev = BEVTransform(bev_config)
geometry_computer = LaneGeometryComputer(bev, config["decision_engine"])
viz_config = VisualizationConfig.from_yaml("configs/lane_detection.yaml")

print(f"Detector ready: {detector.is_ready()}")
print(f"BEV matrix shape: {bev.get_transform_matrix().shape}")
warmup_ms = detector.warmup()
print(f"Warmup inference: {warmup_ms:.1f}ms")

## BEV Transform Calibration Check

In [ ]:
all_images = sorted(glob.glob("rgb/rgb_image_*.png"))
print(f"Found {len(all_images)} images in rgb/")

if not all_images:
    raise FileNotFoundError("No images found in rgb/. Expected rgb/rgb_image_*.png")

first_img = cv2.imread(all_images[0])
bev_out = bev.transform_image(first_img)

# Draw calibration trapezoid on the camera frame
src_pts = np.array(bev_config.src_points, dtype=np.int32)
orig_annotated = first_img.copy()
cv2.polylines(orig_annotated, [src_pts], isClosed=True, color=(0, 0, 255), thickness=2)
for pt in src_pts:
    cv2.circle(orig_annotated, tuple(pt.tolist()), 6, (0, 255, 0), -1)

fig, axes = plt.subplots(1, 2, figsize=(16, 10))
axes[0].imshow(cv2.cvtColor(orig_annotated, cv2.COLOR_BGR2RGB))
axes[0].set_title("Camera View (with trapezoid src points)", fontsize=12)
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(bev_out, cv2.COLOR_BGR2RGB))
axes[1].set_title("Bird's Eye View", fontsize=12)
axes[1].axis("off")

plt.tight_layout()
plt.savefig("outputs/bev_calibration.png", dpi=100, bbox_inches="tight")
plt.show()

print("If the BEV looks like a top-down road, calibration is correct.")
print("If it looks distorted, update bev_transform.src_points in configs/lane_detection.yaml")

## 10 Easy Cases (Clear Road, Good Lighting)

In [ ]:
easy_images = all_images[:10]
results_easy = []

for img_path in easy_images:
    image = cv2.imread(img_path)
    detection = detector.predict(image)
    geometry = geometry_computer.compute(detection)
    viz = create_full_visualization(image, detection, geometry, config=viz_config)
    results_easy.append((img_path, detection, geometry, viz))

fig, axes = plt.subplots(2, 5, figsize=(16, 10))
for idx, (img_path, detection, geometry, viz) in enumerate(results_easy):
    ax = axes[idx // 5][idx % 5]
    ax.imshow(cv2.cvtColor(viz, cv2.COLOR_BGR2RGB))
    n_lanes = int(len(detection.left_lane) > 0) + int(len(detection.right_lane) > 0)
    ax.set_title(f"{Path(img_path).name}\n{n_lanes} lane(s)", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.savefig("outputs/easy_cases.png", dpi=100, bbox_inches="tight")
plt.show()

# Summary
two_plus_easy = sum(
    1 for _, d, _, _ in results_easy
    if len(d.left_lane) > 0 and len(d.right_lane) > 0
)
avg_conf_easy = float(np.mean([
    float(np.mean(d.confidence)) if d.confidence else 0.0
    for _, d, _, _ in results_easy
]))
avg_ms_easy = float(np.mean([d.inference_time_ms for _, d, _, _ in results_easy]))

print(f"Easy cases — images with 2+ lanes : {two_plus_easy}/{len(easy_images)}")
print(f"Easy cases — avg confidence       : {avg_conf_easy:.3f}")
print(f"Easy cases — avg inference time   : {avg_ms_easy:.1f} ms")

## 10 Medium Cases (Shadows, Partial Occlusion)

In [ ]:
medium_images = all_images[10:20]
results_medium = []

for img_path in medium_images:
    image = cv2.imread(img_path)
    detection = detector.predict(image)
    geometry = geometry_computer.compute(detection)
    viz = create_full_visualization(image, detection, geometry, config=viz_config)
    results_medium.append((img_path, detection, geometry, viz))

fig, axes = plt.subplots(2, 5, figsize=(16, 10))
for idx, (img_path, detection, geometry, viz) in enumerate(results_medium):
    ax = axes[idx // 5][idx % 5]
    ax.imshow(cv2.cvtColor(viz, cv2.COLOR_BGR2RGB))
    n_lanes = int(len(detection.left_lane) > 0) + int(len(detection.right_lane) > 0)
    ax.set_title(f"{Path(img_path).name}\n{n_lanes} lane(s)", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.savefig("outputs/medium_cases.png", dpi=100, bbox_inches="tight")
plt.show()

# Summary
two_plus_medium = sum(
    1 for _, d, _, _ in results_medium
    if len(d.left_lane) > 0 and len(d.right_lane) > 0
)
avg_conf_medium = float(np.mean([
    float(np.mean(d.confidence)) if d.confidence else 0.0
    for _, d, _, _ in results_medium
]))
avg_ms_medium = float(np.mean([d.inference_time_ms for _, d, _, _ in results_medium]))
conf_drop = (avg_conf_easy - avg_conf_medium) / max(avg_conf_easy, 1e-9) * 100

print(f"Medium cases — images with 2+ lanes : {two_plus_medium}/{len(medium_images)}")
print(f"Medium cases — avg confidence       : {avg_conf_medium:.3f}")
print(f"Medium cases — avg inference time   : {avg_ms_medium:.1f} ms")
print(f"Confidence drop from easy to medium cases: {conf_drop:.1f}%")

## 5 Hard Cases (Heavy Shadows, Faded Markings, Curves)

In [ ]:
hard_images = all_images[20:25]
results_hard = []

for img_path in hard_images:
    image = cv2.imread(img_path)
    detection = detector.predict(image)
    geometry = geometry_computer.compute(detection)
    viz = create_full_visualization(image, detection, geometry, config=viz_config)
    results_hard.append((img_path, detection, geometry, viz))

n_hard = len(results_hard)
fig, axes = plt.subplots(1, max(n_hard, 1), figsize=(16, 10))
if n_hard == 1:
    axes = [axes]

for idx, (img_path, detection, geometry, viz) in enumerate(results_hard):
    axes[idx].imshow(cv2.cvtColor(viz, cv2.COLOR_BGR2RGB))
    n_lanes = int(len(detection.left_lane) > 0) + int(len(detection.right_lane) > 0)
    axes[idx].set_title(f"{Path(img_path).name}\n{n_lanes} lane(s)", fontsize=8)
    axes[idx].axis("off")
    print(f"\n[{Path(img_path).name}]  {geometry.describe()}")

plt.tight_layout()
plt.savefig("outputs/hard_cases.png", dpi=100, bbox_inches="tight")
plt.show()

## BEV + Polynomial Fitting Validation

In [ ]:
# Find up to 3 images where both lanes were detected
two_lane_samples = []
for img_path in all_images:
    if len(two_lane_samples) >= 3:
        break
    image = cv2.imread(img_path)
    detection = detector.predict(image)
    if len(detection.left_lane) > 0 and len(detection.right_lane) > 0:
        two_lane_samples.append((img_path, image, detection))

print(f"Found {len(two_lane_samples)} two-lane images for BEV polynomial visualisation.")
if not two_lane_samples:
    print("WARNING: No images with both lanes detected — using first 3 images instead.")
    for img_path in all_images[:3]:
        image = cv2.imread(img_path)
        detection = detector.predict(image)
        two_lane_samples.append((img_path, image, detection))

n_rows = len(two_lane_samples)
fig, axes = plt.subplots(n_rows, 3, figsize=(16, 10))
if n_rows == 1:
    axes = [axes]

for row, (img_path, image, detection) in enumerate(two_lane_samples):
    bev_frame = bev.transform_image(image)
    geometry = geometry_computer.compute(detection)

    bev_annotated = bev_frame.copy()
    ys = np.linspace(0, bev_config.output_height - 1, 100)

    for lane_pts, color in [
        (detection.left_lane,  (0, 255, 0)),
        (detection.right_lane, (0, 200, 255)),
    ]:
        if len(lane_pts) < 5:
            continue
        bev_pts = np.array(bev.transform_points(lane_pts), dtype=np.float32)
        if len(bev_pts) < 5:
            continue
        coeffs = np.polyfit(bev_pts[:, 1], bev_pts[:, 0], deg=2)
        xs = np.polyval(coeffs, ys).astype(np.int32)
        ys_int = ys.astype(np.int32)
        pts = np.stack([xs, ys_int], axis=1)
        valid = (pts[:, 0] >= 0) & (pts[:, 0] < bev_config.output_width)
        cv2.polylines(bev_annotated, [pts[valid].reshape(-1, 1, 2)], False, color, 2)

    # Col 0 — camera view with full overlay
    cam_viz = create_full_visualization(image, detection, geometry, config=viz_config)
    axes[row][0].imshow(cv2.cvtColor(cam_viz, cv2.COLOR_BGR2RGB))
    axes[row][0].set_title(f"Camera: {Path(img_path).name}", fontsize=8)
    axes[row][0].axis("off")

    # Col 1 — BEV with polynomial lanes drawn
    axes[row][1].imshow(cv2.cvtColor(bev_annotated, cv2.COLOR_BGR2RGB))
    axes[row][1].set_title("BEV with Polynomial Lanes", fontsize=8)
    axes[row][1].axis("off")

    # Col 2 — geometry stats as a text panel
    axes[row][2].axis("off")
    stats = (
        f"Geometry Stats\n"
        f"{'─' * 22}\n"
        f"Left detected : {geometry.left_lane_detected}\n"
        f"Right detected: {geometry.right_lane_detected}\n"
        f"Offset        : {geometry.offset_m:+.3f} m\n"
        f"Curvature     : {geometry.curvature_inv_m:+.4f} m\u207b\u00b9\n"
        f"Road width    : {geometry.road_width_m:.2f} m\n"
        f"Left conf     : {geometry.left_lane_confidence:.2f}\n"
        f"Right conf    : {geometry.right_lane_confidence:.2f}\n"
        f"\n{geometry.describe()}"
    )
    axes[row][2].text(
        0.05, 0.95, stats,
        transform=axes[row][2].transAxes,
        fontsize=9, verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8),
    )

plt.tight_layout()
plt.savefig("outputs/bev_polynomial.png", dpi=100, bbox_inches="tight")
plt.show()

## Offset and Curvature Analysis

In [ ]:
all_results = []
for img_path in all_images:
    image = cv2.imread(img_path)
    detection = detector.predict(image)
    geometry = geometry_computer.compute(detection)
    all_results.append(geometry)

valid_results     = [g for g in all_results if g.lane_geometry_valid]
both_lane_results = [g for g in all_results if g.both_lanes_detected()]

offsets    = [g.offset_m       for g in valid_results]
curvatures = [g.curvature_inv_m for g in valid_results]
widths     = [g.road_width_m   for g in valid_results if g.road_width_m > 0]

fig, axes = plt.subplots(1, 3, figsize=(16, 10))

axes[0].hist(offsets, bins=20, color="steelblue", edgecolor="white")
axes[0].set_title("Lateral Offset Distribution", fontsize=11)
axes[0].set_xlabel("Offset (m)")
axes[0].set_ylabel("Count")
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.5, label="lane centre")
axes[0].legend()

axes[1].hist(curvatures, bins=20, color="darkorange", edgecolor="white")
axes[1].set_title("Road Curvature Distribution", fontsize=11)
axes[1].set_xlabel("Curvature (m\u207b\u00b9)")
axes[1].set_ylabel("Count")
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.5, label="straight road")
axes[1].legend()

axes[2].hist(widths, bins=20, color="mediumseagreen", edgecolor="white")
axes[2].set_title("Road Width Distribution", fontsize=11)
axes[2].set_xlabel("Width (m)")
axes[2].set_ylabel("Count")
axes[2].axvline(3.5, color="red", linestyle="--", linewidth=1.5, label="std 3.5 m")
axes[2].legend()

plt.tight_layout()
plt.savefig("outputs/geometry_histograms.png", dpi=100, bbox_inches="tight")
plt.show()

n_both   = len(both_lane_results)
n_total  = len(all_images)
mean_off = float(np.mean(offsets)) if offsets else 0.0
std_off  = float(np.std(offsets))  if offsets else 0.0
mean_curv = float(np.mean(curvatures)) if curvatures else 0.0

print(f"Images with both lanes : {n_both}/{n_total}")
print(f"Mean offset            : {mean_off:+.3f} m,  Std: {std_off:.3f} m")
print(f"Mean curvature         : {mean_curv:+.5f} m\u207b\u00b9")

## Inference Latency Benchmark

In [ ]:
benchmark_images = random.sample(all_images, min(20, len(all_images)))
latencies = []

for img_path in benchmark_images:
    image = cv2.imread(img_path)
    result = detector.predict(image)
    latencies.append(result.inference_time_ms)

latencies_sorted = sorted(latencies)
p50  = float(np.percentile(latencies_sorted, 50))
p95  = float(np.percentile(latencies_sorted, 95))
p_max = float(max(latencies_sorted))

fig, ax = plt.subplots(figsize=(16, 10))
ax.hist(latencies_sorted, bins=15, color="mediumpurple", edgecolor="white")
ax.axvline(p50, color="green",  linestyle="--", linewidth=2,   label=f"P50 = {p50:.1f} ms")
ax.axvline(p95, color="orange", linestyle="--", linewidth=2,   label=f"P95 = {p95:.1f} ms")
ax.axvline(30,  color="blue",   linestyle=":",  linewidth=1.5, label="Target 30 ms")
ax.axvline(100, color="red",    linestyle=":",  linewidth=1.5, label="Limit  100 ms")
ax.set_title("UFLDv2 Inference Latency  (CPU · ONNX ResNet-18)", fontsize=12)
ax.set_xlabel("Inference time (ms)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/latency_histogram.png", dpi=100, bbox_inches="tight")
plt.show()

print(f"P50 latency : {p50:.1f} ms")
print(f"P95 latency : {p95:.1f} ms")
print(f"Max latency : {p_max:.1f} ms")
print(f"Target: < 30 ms CPU  |  Actual P95: {p95:.1f} ms")

if p95 > 100:
    print("\u274c FAIL: P95 latency exceeds 100 ms hard limit!")
elif p95 > 30:
    print("\u26a0\ufe0f  WARN: P95 latency exceeds 30 ms target (still within 100 ms limit)")
else:
    print("\u2705 PASS: P95 latency within 30 ms target")

## Phase 2 Exit Criteria Checklist

In [ ]:
checklist = {}

# ── 1. Detector ran on all images without crashing ─────────────────────────
try:
    assert len(all_results) == len(all_images), (
        f"Processed {len(all_results)} / {len(all_images)} images"
    )
    checklist[1] = True
    print(f"\u2705  1. Detector ran on all {len(all_images)} images without crashing")
except Exception as e:
    checklist[1] = False
    print(f"\u274c  1. Detector ran without crashing — {e}")

# ── 2. At least 30 % had 2+ lanes ──────────────────────────────────────────
try:
    n_two_plus = sum(1 for g in all_results if g.both_lanes_detected())
    pct = n_two_plus / max(len(all_images), 1) * 100
    assert pct >= 30, f"Only {pct:.1f}% had 2+ lanes (threshold: 30%)"
    checklist[2] = True
    print(f"\u2705  2. \u226530% images with both lanes detected ({pct:.1f}%  [{n_two_plus}/{len(all_images)}])")
except AssertionError as e:
    checklist[2] = False
    print(f"\u274c  2. \u226530% images with both lanes — {e}")

# ── 3. P95 inference latency < 100 ms ──────────────────────────────────────
try:
    assert p95 < 100, f"P95 = {p95:.1f} ms  (limit: 100 ms)"
    checklist[3] = True
    print(f"\u2705  3. P95 inference latency < 100 ms  (actual: {p95:.1f} ms)")
except AssertionError as e:
    checklist[3] = False
    print(f"\u274c  3. P95 inference latency — {e}")

# ── 4. BEV transform matrix is 3×3 ─────────────────────────────────────────
try:
    mat = bev.get_transform_matrix()
    assert mat.shape == (3, 3), f"Expected (3, 3), got {mat.shape}"
    checklist[4] = True
    print(f"\u2705  4. BEV transform matrix is 3\u00d73  {mat.shape}")
except AssertionError as e:
    checklist[4] = False
    print(f"\u274c  4. BEV transform matrix shape — {e}")

# ── 5. At least one image has valid offset + curvature ─────────────────────
try:
    valid_geom = [
        g for g in all_results
        if g.lane_geometry_valid and abs(g.offset_m) < 10.0 and abs(g.curvature_inv_m) < 1.0
    ]
    assert len(valid_geom) >= 1, "No image produced valid offset + curvature"
    checklist[5] = True
    print(f"\u2705  5. Valid offset + curvature in at least 1 image  ({len(valid_geom)} total)")
except AssertionError as e:
    checklist[5] = False
    print(f"\u274c  5. Valid offset + curvature — {e}")

# ── 6. Save 5 sample visualisations to outputs/ ────────────────────────────
try:
    sample_srcs = all_images[:5] if len(all_images) >= 5 else all_images
    saved = []
    for n, img_path in enumerate(sample_srcs):
        image = cv2.imread(img_path)
        detection = detector.predict(image)
        geometry = geometry_computer.compute(detection)
        viz = create_full_visualization(image, detection, geometry, config=viz_config)
        out_path = f"outputs/phase2_sample_{n + 1}.png"
        cv2.imwrite(out_path, viz)
        saved.append(out_path)
    assert all(Path(p).exists() and Path(p).stat().st_size > 0 for p in saved)
    checklist[6] = True
    print(f"\u2705  6. {len(saved)} sample visualisations saved to outputs/")
except Exception as e:
    checklist[6] = False
    print(f"\u274c  6. Save visualisations — {e}")

# ── Verdict ─────────────────────────────────────────────────────────────────
passed = sum(checklist.values())
total  = len(checklist)
bar    = "=" * 52
print(f"\n{bar}")
print(f"  Phase 2 Exit Criteria: {passed}/{total} PASSED")
if passed == total:
    print("  \U0001f389 ALL CHECKS PASSED — Phase 2 complete, ready for Phase 3.")
else:
    failed = [k for k, v in checklist.items() if not v]
    print(f"  \u26a0\ufe0f  Failed checks: {failed}")
    print("  Resolve the failures above before proceeding to Phase 3.")
print(bar)